# 🧩 31 · Batch effects, data integration and reference mapping
### Seeing batch effects and confounding, scoring integration against ground truth, four corrections built from scratch (ComBat-style, mutual nearest neighbours, Harmony-lite, a batch-aware VAE) and where each over-corrects, and mapping a new sample onto a reference atlas with honest confidence

**BiologyToML** · Level 2 · Research frontier · Part G — Cells and tissues · Module 31 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/31_batch_integration.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, Matplotlib, scikit-learn, PyTorch (CPU) |
| **Prerequisites** | Level 1 (especially Modules 06, 08, 12–14 and 21); Modules 25 (variational autoencoders) and 30 (single-cell RNA-seq basics) — the pieces we need are re-explained here |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [06 · Linear regression](06_linear_regression_gwas.ipynb), [08 · k-NN](08_knn_naive_bayes.ipynb), [12 · Clustering](12_clustering_kmeans_hierarchical.ipynb), [13 · Mixture models and EM](13_mixture_models_em.ipynb), [14 · PCA](14_pca_dimensionality_reduction.ipynb), [21 · Bayesian learning](21_bayesian_learning_mcmc.ipynb), [25 · Variational inference and VAEs](25_variational_inference_vae.ipynb), [30 · Single-cell RNA-seq](30_single_cell_rnaseq.ipynb) |
| **Research frontier covered** | batch correction (ComBat, MNN, Harmony, scVI), benchmarking integration, atlas building and reference mapping — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED** with `simulate_scrna`: PBMC-like UMI counts from 6 donors (3 control, 3 interferon-stimulated)
> processed in 2–3 batches, with the true cell type, donor, batch and condition of every cell and the true stimulation
> effects known. Three designs are simulated: **balanced** (every batch has every type and both conditions), **missing
> type** (no B cells in batch 2) and **confounded** (all stimulated donors in one batch). In this simulator a batch effect
> is a per-gene shift shared by all cell types plus a depth change — exactly what linear methods assume, which flatters
> them; real batch effects are often cell-type-specific (see 🧭 *From toy to real data*).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** where batch effects come from, **recognise** them in a PCA, and **prove** with a design matrix why a confounded design (batch = condition) cannot be rescued by any correction method.
2. **Implement** integration metrics that use ground truth — batch mixing (a type-aware neighbourhood entropy and a kBET-like test) and biology conservation (ARI, NMI, silhouette, k-NN accuracy) — and **explain** why both are needed.
3. **Implement** per-gene centring and a ComBat-style empirical-Bayes location/scale correction, and **demonstrate** over-correction when cell-type composition differs between batches.
4. **Implement** mutual-nearest-neighbour (MNN) correction and a simplified Harmony (soft k-means with a diversity penalty plus per-cluster linear correction), and **check** their intermediate steps against the truth.
5. **Train** the batch-conditioned negative-binomial VAE of Module 25 as an integration method.
6. **Benchmark** all methods on a balanced and a missing-type design in the style of scIB, and **discuss** the trade-off between removing batch effects and preserving biology.
7. **Map** a new sample onto a reference by projection, **transfer** cell-type labels with a k-NN confidence score, **detect** a cell type absent from the reference (AUROC), and **show** how forced integration can hide novel cells.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, course toolkit | 5 min |
| 1 · Seeing batch effects | PCA by batch and type; a confounded design and identifiability | 20 min |
| 2 · Measuring success with ground truth | mixing and conservation metrics; why both | 20 min |
| 3 · Linear correction | centring, ComBat-style empirical Bayes; over-correction with a missing type | 25 min |
| 4 · Mutual nearest neighbours | MNN pairs, smoothed correction vectors | 20 min |
| 5 · Harmony-lite | soft k-means with a diversity penalty + per-cluster correction | 20 min |
| 6 · A batch-conditioned VAE | Module 25's mini-scVI as an integration method | 15 min |
| 7 · 🧪 Benchmark | all methods, two designs, mixing vs conservation | 15 min |
| 8 · Reference mapping and label transfer | projection, k-NN labels, confidence, unknown types | 25 min |
| 9 · 🧭 From toy to real data | Harmony, scVI/scANVI, Seurat, scArches, atlases | 5 min |
| ✅ / 🧠 / 🚀 / 📝 | checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

The Human Cell Atlas and the CELLxGENE collection hold single-cell profiles from thousands of donors, measured by hundreds
of laboratories with different dissociation protocols, chemistries, sequencers and processing days. Integrated atlases of
the lung, the immune system or the developing brain combine dozens of such data sets so that a cell type can be studied
across people, ages and diseases. But every sample carries a **technical fingerprint** — a *batch effect* — that shifts
the expression of many genes at once. Analysed naively, cells cluster by *when and where they were processed* rather than
by *what they are*.

Batch correction ("data integration") tries to remove the technical fingerprint **without erasing biology**. That is
harder than it sounds: a method that simply forces batches to overlap will also merge a cell type that exists in only one
batch, or erase a disease effect that happens to coincide with a batch. And once an atlas exists, a new study wants to
**map** its cells onto it — to borrow the atlas's carefully curated cell-type labels — while still noticing cells the
atlas has never seen (a new cell state in a patient, say).

This notebook builds the main families of methods from scratch and judges them where it is possible: on **simulated**
data with a known truth. The design mimics a common PBMC study — six donors, three control and three stimulated with
interferon-β as in Kang *et al.* (2018), processed in three batches — with variants in which one cell type is missing
from a batch, or in which batch and condition coincide.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import sparse, stats
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, roc_auc_score, silhouette_score
from sklearn.neighbors import NearestNeighbors

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration (used in Section 6) ──
torch.manual_seed(SEED)
torch.set_num_threads(1)                 # tiny networks: one thread is about as fast and avoids contention
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_scrna` generates PBMC-like UMI counts with known cell types, donors, batches and stimulation; `log_normalize` and `pca` are the standard preprocessing; `plot_embedding` draws 2-D maps of cells. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MITO_GENES = ["MT-ND1", "MT-ND2", "MT-CO1", "MT-CO2", "MT-ATP8", "MT-ATP6", "MT-CO3", "MT-ND3",
              "MT-ND4L", "MT-ND4", "MT-ND5", "MT-ND6", "MT-CYB"]


INTERFERON_GENES = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "IFI44L", "RSAD2", "IRF7", "STAT1"]


CYCLING_GENES = ["MKI67", "TOP2A", "CCNB1", "CDK1", "UBE2C", "BIRC5", "TYMS", "PCNA"]


PBMC_TYPES = {
    "CD4 T": ("lymphoid", ["CD3E", "CD3D", "IL7R", "CD4"], 0.5),
    "CD8 T": ("lymphoid", ["CD3E", "CD3D", "CD8A", "CD8B"], 0.5),
    "NK": ("lymphoid", ["NKG7", "GNLY", "KLRD1", "PRF1"], 2.2),
    "B": ("lymphoid", ["MS4A1", "CD79A", "CD79B", "CD19"], 2.4),
    "Monocyte": ("myeloid", ["CD14", "LYZ", "S100A8", "S100A9"], 2.6),
    "Dendritic": ("myeloid", ["FCER1A", "CST3", "CLEC10A", "HLA-DQA1"], 2.2),
}


def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


def simulate_scrna(rng: np.random.Generator, n_cells: int = 3000, n_genes: int = 1000, cell_types=None,
                   proportions=None, n_donors: int = 6, n_batches: int = 2, confounded: bool = False,
                   depth: float = 1500, batch_effect: float = 0.25, donor_effect: float = 0.15,
                   stim_effect: float = 1.0, theta: float = 8.0, doublet_rate: float = 0.0,
                   damaged_rate: float = 0.0, cycling_rate: float = 0.0, drop_type_in_batch=None) -> dict:
    """Simulate single-cell RNA-seq UMI counts for a PBMC-like sample collection.

    Design: ``n_donors`` donors, the first half *control* and the second half *stimulated*
    (an interferon response in every cell type plus a monocyte-specific response).  Donor
    ``d`` is sequenced in batch ``d % n_batches`` — or, with ``confounded=True``, every control
    donor in batch 1 and every stimulated donor in batch 2 (a design flaw to learn from).
    Counts are negative binomial around ``depth`` UMIs per cell.  Optional artefacts for
    quality control: doublets, damaged cells (high mitochondrial fraction), cycling cells.
    ``drop_type_in_batch=("B", 1)`` removes B cells from batch 1 (index 1, i.e. "batch2").

    Expression probabilities are renormalised within each cell (a fixed-size library, as in
    sequencing), so in stimulated cells the genes with ``true_logfc == 0`` lose a few per cent of
    their share (about −0.04 on the natural-log scale) to the induced genes: a compositional
    effect that is also present in real data.

    Returns a dict with ``counts`` (cells × genes, int32), ``obs`` (per-cell table with the
    true cell type, donor, batch, condition, …), ``var`` (per-gene table), ``true_logfc``
    (cell type × gene stimulation effect, natural log) and ``log_profiles``.
    """
    cell_types = PBMC_TYPES if cell_types is None else cell_types
    names = list(cell_types)
    T = len(names)
    p = np.full(T, 1 / T) if proportions is None else np.asarray(proportions, float) / np.sum(proportions)
    extra = MITO_GENES + INTERFERON_GENES + CYCLING_GENES
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, cell_types, extra_names=extra)
    G = len(genes)
    is_mito, is_ifn, is_cyc = (np.isin(genes, MITO_GENES), np.isin(genes, INTERFERON_GENES),
                               np.isin(genes, CYCLING_GENES))
    profiles[:, is_mito] = rng.normal(3.0, 0.3, is_mito.sum())          # ≈ 4 % of UMIs are mitochondrial
    profiles[:, is_ifn | is_cyc] = rng.normal(-1.0, 0.4, (T, (is_ifn | is_cyc).sum()))
    # stimulation: interferon genes up everywhere + 40 monocyte-specific responders (up or down)
    true_logfc = np.zeros((T, G))
    true_logfc[:, is_ifn] = stim_effect * rng.normal(1.6, 0.3, is_ifn.sum())
    if "Monocyte" in names:
        responders = rng.choice(np.flatnonzero((marker_of == "") & ~is_mito & ~is_ifn & ~is_cyc), 40, replace=False)
        true_logfc[names.index("Monocyte"), responders] = stim_effect * rng.choice([-1, 1], 40) * rng.normal(1.2, 0.3, 40)
    # donors, batches, conditions
    donor = rng.integers(0, n_donors, n_cells)
    stimulated = donor >= n_donors // 2
    batch = stimulated.astype(int) if confounded else donor % n_batches
    batch = np.minimum(batch, n_batches - 1)
    cell_type = rng.choice(T, n_cells, p=p)
    if drop_type_in_batch is not None:
        t_drop, b_drop = names.index(drop_type_in_batch[0]), drop_type_in_batch[1]
        bad = (cell_type == t_drop) & (batch == b_drop)
        cell_type[bad] = rng.choice(np.delete(np.arange(T), t_drop), bad.sum(), p=np.delete(p, t_drop) / (1 - p[t_drop]))
    batch_shift = rng.normal(0, batch_effect, (n_batches, G))
    batch_depth = np.exp(rng.normal(0, 0.3, n_batches))
    donor_shift = rng.normal(0, donor_effect, (n_donors, G))
    log_rate = (profiles[cell_type] + batch_shift[batch] + donor_shift[donor]
                + true_logfc[cell_type] * stimulated[:, None])
    cycling = rng.random(n_cells) < cycling_rate
    log_rate[np.ix_(cycling, np.flatnonzero(is_cyc))] += 3.0
    damaged = rng.random(n_cells) < damaged_rate
    log_rate[np.ix_(damaged, np.flatnonzero(is_mito))] += 2.2
    lib = depth * batch_depth[batch] * np.exp(rng.normal(0, 0.35, n_cells))
    lib[damaged] *= 0.35
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    theta_g = theta * np.exp(rng.normal(0, 0.5, G))
    mu = lib[:, None] * prob
    doublet = rng.random(n_cells) < doublet_rate
    partner = rng.integers(0, n_cells, n_cells)
    mu[doublet] = 0.9 * mu[doublet] + 0.9 * mu[partner[doublet]]       # two cells in one droplet
    counts = sample_negative_binomial(rng, mu, theta_g)
    obs = pd.DataFrame({
        "cell_type": np.where(doublet, "Doublet", np.array(names, dtype=object)[cell_type]),
        "true_type": np.array(names, dtype=object)[cell_type],
        "lineage": np.array([cell_types[n][0] for n in names], dtype=object)[cell_type],
        "donor": [f"donor{d + 1}" for d in donor],
        "condition": np.where(stimulated, "stimulated", "control"),
        "batch": [f"batch{b + 1}" for b in batch],
        "n_counts": counts.sum(axis=1),
        "n_genes": (counts > 0).sum(axis=1),
        "pct_mito": 100 * counts[:, is_mito].sum(axis=1) / np.maximum(counts.sum(axis=1), 1),
        "is_doublet": doublet, "is_damaged": damaged, "is_cycling": cycling,
    }, index=[f"cell{i:05d}" for i in range(n_cells)])
    var = pd.DataFrame({"marker_of": marker_of, "is_mito": is_mito, "is_interferon": is_ifn,
                        "is_cycling": is_cyc, "theta": theta_g}, index=genes)
    return dict(counts=counts, obs=obs, var=var, log_profiles=pd.DataFrame(profiles, index=names, columns=genes),
                true_logfc=pd.DataFrame(true_logfc, index=names, columns=genes))


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['sample_negative_binomial', 'make_expression_programs', 'simulate_scrna', 'log_normalize', 'pca', 'plot_embedding']))

---
## 1 · Seeing batch effects

> 📍 **Where we are.** [Module 30](30_single_cell_rnaseq.ipynb) took one sample from counts to cell types; [Module 25 §7](25_variational_inference_vae.ipynb)
> showed a VAE learning to ignore a batch. Here we look at batch effects themselves: how big they are, how they show up
> in PCA ([Module 14](14_pca_dimensionality_reduction.ipynb)), and the one situation in which nothing can fix them.

🔬 **Assay box — where batch effects come from.** A *batch* is a group of samples processed together: the same day,
operator, reagent lot, 10x chip, library preparation and sequencing run. Differences between batches in capture
efficiency, reverse-transcription and amplification bias, sequencing depth, ambient RNA and dissociation stress change the
measured counts of many genes at once — often by tens of percent, sometimes by much more (different chemistries or
single-nucleus vs single-cell protocols). In a well-designed study, every batch contains samples of every condition, so
that batch and biology can be told apart.

**What the simulator does (the ground truth).** For cell $c$ of type $t$ from donor $d$ in batch $b$, the log expression
rate of gene $g$ is

$$\log\lambda_{cg}=\underbrace{\text{profile}_{tg}}_{\text{cell type}}+\underbrace{\gamma_{bg}}_{\text{batch shift}}
+\underbrace{\eta_{dg}}_{\text{donor}}+\underbrace{\beta_{tg}\,\mathbb 1[\text{stimulated}]}_{\text{interferon response}},$$

*In words:* each gene's expression is the cell type's baseline, plus a batch-specific shift (drawn once per batch and gene
with standard deviation `batch_effect` = 0.5 on the natural-log scale, so a typical gene changes by a factor between
about 0.6 and 1.65), plus a smaller
donor shift, plus the stimulation effect for stimulated donors. Each batch also has its own sequencing depth. Counts are
then drawn from a negative binomial around these rates. Note that the batch shift $\gamma_{bg}$ is **the same for every
cell type** — the simplest kind of batch effect, and the one linear methods assume.

In [ ]:
# ── 1.1 SIMULATED PBMC-like data: 6 donors processed in 3 batches (balanced design) ──
rng = np.random.default_rng(SEED + 1)
sim_bal = simulate_scrna(rng, n_cells=4000, n_genes=1000, n_batches=3, n_donors=6, batch_effect=0.5)


def preprocess(sim, n_hvg=500, n_pcs=20):
    """Log-normalise, keep the most variable genes; return (log data, raw counts, gene indices, PCA scores)."""
    lognorm_all = log_normalize(sim["counts"])
    genes = np.argsort(lognorm_all.var(axis=0))[::-1][:n_hvg]
    X = lognorm_all[:, genes].astype(np.float32)
    return X, sim["counts"][:, genes].astype(np.float32), genes, pca(X, n_pcs)[0]


X_bal, C_bal, genes_bal, P_bal = preprocess(sim_bal)
batch_bal = sim_bal["obs"]["batch"].to_numpy(str)
type_bal = sim_bal["obs"]["true_type"].to_numpy(str)
print(f"SIMULATED: {X_bal.shape[0]} cells × {X_bal.shape[1]} highly variable genes; 20 principal components kept")
print("donors per batch:", {b: sorted(set(d)) for b, d in sim_bal["obs"].groupby("batch")["donor"]})
print(pd.crosstab(sim_bal["obs"]["batch"], sim_bal["obs"]["condition"]).to_string())

In [ ]:
# ── 1.2 Which principal components describe batch, which describe biology? ──
def r2_of_labels(scores, labels):
    """Fraction of the variance of each column of `scores` explained by a categorical label (one-way ANOVA R²)."""
    total = ((scores - scores.mean(axis=0)) ** 2).sum(axis=0)
    within = sum(((scores[labels == l] - scores[labels == l].mean(axis=0)) ** 2).sum(axis=0) for l in np.unique(labels))
    return 1 - within / total


r2 = pd.DataFrame({"R² by batch": r2_of_labels(P_bal[:, :6], batch_bal),
                   "R² by true cell type": r2_of_labels(P_bal[:, :6], type_bal)},
                  index=[f"PC{i}" for i in range(1, 7)])
print(r2.round(2).to_string())
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
plot_embedding(P_bal[:, :2], batch_bal, ax=a1, title="PC1 vs PC2 coloured by batch")
plot_embedding(P_bal[:, :2], type_bal, ax=a2, title="the same cells coloured by true type")
fig.suptitle("SIMULATED balanced design: the largest axes of variation are technical", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The two largest principal components are almost entirely **batch**: the batch label explains
92 % and 96 % of the variance of PC1 and PC2 (cell type about 1 %), while the true cell types take over from PC3 onwards
(R² 0.82–0.94). In the picture, cells form three islands — one per batch — and every cell type appears in
each of them. A clustering of these data would discover the processing days before the immune cell types. This is not
an exaggeration: with 500 genes each shifted by a random factor (typically 0.6–1.65), the batch difference adds up over
many genes, whereas types differ strongly only in their marker genes.

The good news is that this design is **balanced**: each batch contains one control and one stimulated donor and all six
cell types, so batch and biology vary independently. Now the bad case.

In [ ]:
# ── 1.3 A confounded design: all control donors in batch 1, all stimulated donors in batch 2 ──
rng = np.random.default_rng(SEED + 2)
sim_conf = simulate_scrna(rng, n_cells=4000, n_genes=1000, n_batches=2, n_donors=6, batch_effect=0.5, confounded=True)
obs_c = sim_conf["obs"]
print(pd.crosstab(obs_c["batch"], obs_c["condition"]).to_string())
design = np.column_stack([np.ones(len(obs_c)), obs_c["batch"] == "batch2", obs_c["condition"] == "stimulated"])
print("design matrix [intercept, batch2, stimulated]: rank", np.linalg.matrix_rank(design.astype(float)), "of 3 columns")


def center_by_batch(X, batches):
    """Per-gene batch-mean centring: subtract each batch's mean of every gene, add back the overall mean."""
    out = X.copy()
    for b in np.unique(batches):
        out[batches == b] = X[batches == b] - X[batches == b].mean(axis=0) + X.mean(axis=0)
    return out


def ifn_response(X, condition, is_ifn):
    """Mean over interferon genes of (stimulated − control) mean log-normalised expression."""
    return float((X[condition == "stimulated"].mean(0) - X[condition == "control"].mean(0))[is_ifn].mean())


rows = {}
for name, sim in [("balanced (3 batches)", sim_bal), ("confounded (2 batches)", sim_conf)]:
    L = log_normalize(sim["counts"])
    cond, bat = sim["obs"]["condition"].to_numpy(str), sim["obs"]["batch"].to_numpy(str)
    is_ifn = sim["var"]["is_interferon"].to_numpy()
    rows[name] = {"true effect (natural log)": sim["true_logfc"].loc["CD4 T", is_ifn].mean(),
                  "estimated, uncorrected": ifn_response(L, cond, is_ifn),
                  "estimated after batch centring": ifn_response(center_by_batch(L, bat), cond, is_ifn)}
print((pd.DataFrame(rows).T.round(3) + 0.0).to_string())    # + 0.0 turns −0.000 into 0.000

🔍 **What just happened?** In the confounded design every control cell is in batch 1 and every stimulated cell in batch 2,
so the "batch" and "stimulated" columns of the design matrix are **identical** and the matrix has rank 2 instead of 3.

> 🔗 **Builds on [Module 06](06_linear_regression_gwas.ipynb).** Least squares needs $X^\top X$ to be invertible, i.e. the
> columns of the design matrix to be linearly independent. Here two columns coincide, so the coefficients of batch and
> stimulation are **not identifiable**: any split of the observed difference between "batch" and "biology" fits the data
> equally well, and no amount of data from this design can tell them apart.

The table shows the consequence for the interferon response (true effect ≈ 1.6 on the natural-log scale in every cell
type; the estimates, about 0.4, are on the log-normalised scale, which compresses changes of lowly expressed genes). In
the balanced design, centring each batch leaves the response intact (0.431 before, 0.433 after), because each batch
contains both conditions. In the confounded design, the uncorrected estimate (0.368) mixes the response with the batch
shift of these genes, and batch centring removes it **completely** (0.000): subtracting the batch means subtracts the
condition means. Any
method that removes "batch" here removes the biology with it — and any method that keeps the biology keeps the batch
effect. **No algorithm can fix a confounded design; only the experimental design can** (randomise samples across batches,
include reference samples, or at least avoid processing all cases on one day and all controls on another).

⚠️ **Partial confounding** (e.g. 80 % of patients in one batch) is common in practice and makes estimates noisy and
model-dependent rather than impossible. Always tabulate batch against every biological variable before analysing.

✏️ **Your turn.** Make the design *partially* confounded: in `sim_bal`, keep all control donors but drop the stimulated
donor of batch 1 (`donor4`). Recompute the interferon response after centring. Is it still close to the balanced value?

---
## 2 · Measuring success with ground truth

> 📍 **Where we are.** Before correcting anything we need a scoreboard. Integration has two goals that pull against each
> other, so every method is scored on both: **batch mixing** (are cells of the same type from different batches
> neighbours?) and **biology conservation** (are different cell types still apart?). Because the data are simulated, we
> know every cell's true type and can score honestly.

**Batch mixing.**

- *Neighbourhood batch entropy* (an iLISI-like score). For each cell, take its $k=30$ nearest neighbours and compute the
  entropy of their batch labels, $H=-\sum_b f_b\log f_b$, where $f_b$ is the fraction of neighbours from batch $b$. We
  divide by the entropy of the batch composition **of the cell's own true type** (and cap at 1), so a perfectly mixed
  neighbourhood scores 1 even when a type is missing from one batch, and a neighbourhood from a single batch scores 0.
- *kBET-like acceptance* (after Büttner *et al.* 2019). For each cell, a $\chi^2$ test compares the observed batch counts
  among its neighbours, $O_b=kf_b$, with the counts expected from its type's composition, $E_b$:
  $\chi^2=\sum_b(O_b-E_b)^2/E_b$. The score is the fraction of cells where the test does **not** reject at 5 %.

**Biology conservation** ([Modules 08](08_knn_naive_bayes.ipynb), [12](12_clustering_kmeans_hierarchical.ipynb), [13](13_mixture_models_em.ipynb)):
ARI and normalised mutual information (NMI) between k-means clusters ($k$ = 6) and the true types, the silhouette of the
true types rescaled to 0–1, and the k-NN accuracy of predicting a cell's type from its neighbours.

Following the scIB benchmark (Luecken *et al.* 2022), we average the batch metrics into a *batch score*, the biology
metrics into a *bio score*, and report $\text{overall}=0.4\times\text{batch}+0.6\times\text{bio}$.

⚠️ **Using the truth is a privilege of simulation.** On real data the "true types" come from expert annotation of each
batch separately, which is itself uncertain — one reason why integration benchmarks are debated.

In [ ]:
# ── 2.1 One scoring function used throughout: batch mixing and biology conservation ──
def shannon(f):
    """Row-wise entropy −Σ f log f (0 log 0 = 0)."""
    return -np.sum(np.where(f > 0, f * np.log(np.where(f > 0, f, 1.0)), 0.0), axis=1)


def integration_scores(Z, batches, types, k=30, seed=SEED):
    """Ground-truth integration metrics for an embedding Z (cells × dimensions); higher is better for all."""
    idx = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z, return_distance=False)[:, 1:]
    b_lev, t_lev = np.unique(batches), np.unique(types)
    frac = np.stack([(batches[idx] == b).mean(axis=1) for b in b_lev], axis=1)       # batch mix of the neighbours
    comp = np.stack([[np.mean(batches[types == t] == b) for b in b_lev] for t in t_lev])
    expect = comp[np.searchsorted(t_lev, types)]                                     # composition of the cell's type
    ok = shannon(expect) > 0                                                         # types present in ≥ 2 batches
    mixing = np.mean(np.minimum(shannon(frac[ok]) / shannon(expect[ok]), 1.0))
    chi2 = k * np.sum(np.where(expect > 0, (frac - expect) ** 2 / np.where(expect > 0, expect, 1.0), 0.0), axis=1)
    kbet = np.mean(stats.chi2.sf(chi2[ok], (expect[ok] > 0).sum(axis=1) - 1) >= 0.05)
    votes = np.stack([(types[idx] == t).sum(axis=1) for t in t_lev], axis=1)
    clusters = KMeans(n_clusters=len(t_lev), n_init=5, random_state=seed).fit_predict(Z)
    s = {"mixing entropy": mixing, "kBET acceptance": kbet,
         "ARI": adjusted_rand_score(types, clusters), "NMI": normalized_mutual_info_score(types, clusters),
         "silhouette (0-1)": (silhouette_score(Z, types) + 1) / 2, "kNN accuracy": np.mean(t_lev[votes.argmax(1)] == types)}
    s["batch score"] = (s["mixing entropy"] + s["kBET acceptance"]) / 2
    s["bio score"] = np.mean([s["ARI"], s["NMI"], s["silhouette (0-1)"], s["kNN accuracy"]])
    s["overall"] = 0.4 * s["batch score"] + 0.6 * s["bio score"]
    return s

In [ ]:
# ── 2.2 Two extreme "integrations": do nothing, or replace every cell by random noise ──
gen = np.random.default_rng(SEED + 22)
baselines = {"uncorrected PCA": P_bal, "random noise": gen.standard_normal(P_bal.shape)}
score_bal = {name: integration_scores(Z, batch_bal, type_bal) for name, Z in baselines.items()}
print(pd.DataFrame(score_bal).T.round(3).to_string())

🔍 **What just happened?** The uncorrected PCA keeps biology locally — a cell's 30 neighbours are of its own type (k-NN
accuracy close to 1) — but its batch mixing is essentially zero, and k-means finds batches instead of types (low ARI).
The random embedding is the mirror image: perfect mixing scores, but no biology at all (ARI and NMI ≈ 0, k-NN accuracy at
chance ≈ 1/6). Its *overall* score (0.49) is even higher than that of the uncorrected data (0.31). This is the central
lesson of integration benchmarking: **a
mixing metric alone rewards destroying information**, so every method must be judged on both axes — and the weights of
the overall score are a convention, not a law of nature.

✏️ **Your turn.** Compute the scores of an embedding that keeps only the true cell-type centroid of every cell
(`Z = pd.DataFrame(P_bal).groupby(type_bal).transform("mean").to_numpy()` plus a little noise). Which metrics does this
"perfect cheat" max out? What does that tell you about using the same labels to correct and to evaluate?

---
## 3 · Linear correction: centring and ComBat

> 📍 **Where we are.** The simplest idea: treat batch as a per-gene offset and subtract it — the regression view of
> [Module 06](06_linear_regression_gwas.ipynb). Its empirical-Bayes refinement, **ComBat** (Johnson, Li & Rabinovic
> 2007), was designed for microarray studies with few samples per batch and is still widely used.

**Per-gene centring** subtracts each batch's mean of each gene (and adds back the overall mean): it removes the term
$\gamma_{bg}$ of Section 1 *if every batch has the same mixture of cell types and conditions*. Otherwise the batch mean
also contains biology, and subtracting it distorts every cell of that batch.

**ComBat** models standardised expression $z_{cg}$ of cell $c$ in batch $b$ as
$z_{cg}=\gamma_{bg}+\delta_{bg}\,\varepsilon_{cg}$ — a batch-specific location $\gamma_{bg}$ and scale $\delta_{bg}$ for each
gene — and estimates them with **empirical Bayes**: the per-gene estimates are shrunk towards what is typical for that
batch across all genes,

$$\gamma^*_{bg}=\frac{n_b\,\tau_b^2\,\hat\gamma_{bg}+\delta^{*2}_{bg}\,\bar\gamma_b}{n_b\,\tau_b^2+\delta^{*2}_{bg}},\qquad
\delta^{*2}_{bg}=\frac{\theta_b+\tfrac12\sum_{c\in b}(z_{cg}-\gamma^*_{bg})^2}{n_b/2+\lambda_b-1}.$$

*In words:* the corrected batch mean of a gene is a precision-weighted compromise between that gene's own batch mean
$\hat\gamma_{bg}$ (trusted more when the batch has many cells $n_b$) and the average shift $\bar\gamma_b$ of all genes in
the batch (whose spread across genes is $\tau_b^2$); the variance is shrunk similarly towards an inverse-gamma prior with
parameters $\lambda_b,\theta_b$ fitted across genes. The corrected value is $(z_{cg}-\gamma^*_{bg})/\delta^*_{bg}$.

> 🔗 **Builds on [Module 21](21_bayesian_learning_mcmc.ipynb).** This is the conjugate-prior posterior mean you derived for a
> Gaussian mean — a compromise between prior and data weighted by their precisions — with the prior's parameters
> estimated from all genes at once ("borrowing strength"). With thousands of cells per batch the data dominate and
> shrinkage is negligible; with 3–5 microarrays per batch it matters a lot.

In [ ]:
# ── 3.1 ComBat-lite: per-gene location/scale adjustment with empirical-Bayes shrinkage ──
def combat_lite(X, batches, n_iter=30):
    """Standardise genes, estimate each batch's per-gene mean γ and variance δ², shrink them towards the batch's
    across-gene average (normal and inverse-gamma priors fitted by moments, as in Johnson et al. 2007), remove them."""
    grand, levels = X.mean(axis=0), np.unique(batches)
    pooled_ss = sum(((X[batches == b] - X[batches == b].mean(axis=0)) ** 2).sum(axis=0) for b in levels)
    sd = np.sqrt(pooled_ss / len(X))
    sd = np.where(sd > 0, sd, 1.0)
    Zs = (X - grand) / sd                                         # standardised (pooled within-batch sd)
    out = np.empty_like(Zs)
    for b in levels:
        Zb, n = Zs[batches == b], np.sum(batches == b)
        g_hat, d_hat = Zb.mean(axis=0), Zb.var(axis=0, ddof=1)   # raw per-gene location and scale of this batch
        g_bar, tau2 = g_hat.mean(), g_hat.var(ddof=1)            # normal prior for γ, fitted across genes
        m, v = d_hat.mean(), d_hat.var(ddof=1)                   # inverse-gamma prior for δ², by moments
        lam, theta = (2 * v + m ** 2) / v, (m * v + m ** 3) / v
        g, d = g_hat.copy(), d_hat.copy()
        for _ in range(n_iter):                                   # the two posterior updates depend on each other
            g = (n * tau2 * g_hat + d * g_bar) / (n * tau2 + d)
            d = (theta + 0.5 * ((Zb - g) ** 2).sum(axis=0)) / (n / 2 + lam - 1)
        out[batches == b] = (Zb - g) / np.sqrt(d)
    return out * sd + grand


def show_embeddings(embs, batches, types, title):
    """2-D PCA view of each embedding: coloured by batch (top row) and by true type (bottom row)."""
    fig, axes = plt.subplots(2, len(embs), figsize=(min(12, 2.3 * len(embs) + 0.6), 5.2), squeeze=False)
    for j, (name, Z) in enumerate(embs.items()):
        xy = pca(Z, 2)[0]
        plot_embedding(xy, batches, ax=axes[0, j], title=f"{name}: batch", legend=(j == 0), s=2)
        plot_embedding(xy, types, ax=axes[1, j], title=f"{name}: type", legend=(j == 0), s=2)
    fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

In [ ]:
# ── 3.2 Centring and ComBat-lite on the balanced design ──
X_cent, X_combat = center_by_batch(X_bal, batch_bal), combat_lite(X_bal, batch_bal)
emb_bal = {"uncorrected": P_bal, "centring": pca(X_cent, 20)[0], "ComBat-lite": pca(X_combat, 20)[0]}
score_bal = {"uncorrected": score_bal["uncorrected PCA"], "random noise": score_bal["random noise"]}
for name in ("centring", "ComBat-lite"):
    score_bal[name] = integration_scores(emb_bal[name], batch_bal, type_bal)
print(pd.DataFrame(score_bal).T.round(3).to_string())
show_embeddings(emb_bal, batch_bal, type_bal, "SIMULATED balanced design: linear corrections")

🔍 **What just happened?** Both linear corrections turn three batch islands into cell-type islands: biology
conservation jumps (ARI from 0.17 and NMI from 0.36 to 0.998 and 0.997) and the mixing entropy rises from 0 to 0.78
(centring) and 0.92 (ComBat-lite). ComBat mixes better than plain centring because it also equalises each gene's *variance* across batches (the batches
differ in sequencing depth, which changes the spread of log-normalised values). On this simulation, where the true batch
effect is exactly a per-gene shift shared by all cell types, linear correction is the *right model* — which is why it
does so well. Now break its key assumption: equal composition.

In [ ]:
# ── 3.3 🧪 Over-correction: the same methods when batch 2 contains no B cells ──
rng = np.random.default_rng(SEED + 3)
sim_mis = simulate_scrna(rng, n_cells=4000, n_genes=1000, n_batches=3, n_donors=6, batch_effect=0.5,
                         drop_type_in_batch=("B", 1))                   # index 1 = "batch2"
X_mis, C_mis, genes_mis, P_mis = preprocess(sim_mis)
batch_mis, type_mis = sim_mis["obs"]["batch"].to_numpy(str), sim_mis["obs"]["true_type"].to_numpy(str)
print("B cells per batch:", pd.Series(batch_mis[type_mis == "B"]).value_counts().reindex(["batch1", "batch2", "batch3"], fill_value=0).to_dict())
versions = {"uncorrected": X_mis, "centring": center_by_batch(X_mis, batch_mis), "ComBat-lite": combat_lite(X_mis, batch_mis)}
is_b_marker = sim_mis["var"]["marker_of"].to_numpy()[genes_mis] == "B"
not_b = type_mis != "B"
table = pd.DataFrame({name: {b: M[not_b & (batch_mis == b)][:, is_b_marker].mean() for b in ["batch1", "batch2", "batch3"]}
                      for name, M in versions.items()}).T
table["excess in batch 2"] = table["batch2"] - (table["batch1"] + table["batch3"]) / 2
print(f"mean log-expression of the {is_b_marker.sum()} B-cell marker genes in cells that are NOT B cells:")
print(table.round(3).to_string())
fig, ax = plt.subplots(figsize=(6.4, 3.2))
table[["batch1", "batch2", "batch3"]].plot.bar(ax=ax, rot=0, color=["tab:blue", "tab:orange", "tab:green"])
ax.set_ylabel("B-marker expression in non-B cells"); ax.legend(fontsize=7, title="batch", title_fontsize=7)
ax.set_title("Correcting with batch means paints a B-cell signature onto batch 2 (SIMULATED)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Before correction, B-marker expression in non-B cells differs between batches only through the
random batch shifts of these genes (batch 1 happens to be lower). After centring or ComBat, batches 1 and 3 agree, as they
should for the same cell types — but the non-B cells of batch 2 (T cells, NK cells, monocytes, dendritic cells) now carry
clearly **more** B-cell marker expression than the same cell types in batches 1 and 3: an excess of 0.34 log-units after
centring and 0.31 after ComBat-lite.
The mechanism is simple: batch 2 has no B cells, so its average B-marker expression is low *for biological reasons*;
the correction treats that difference as technical and adds it back to every cell of batch 2. Nothing is wrong with the
arithmetic — the **assumption of equal composition** is violated.

This is **over-correction**: real biology (a composition difference) has been removed and replaced by an artefact (a
spurious B-cell programme in other cells). It can create false "differentially expressed genes" in downstream analyses.
ComBat can protect known biological covariates by including them in its model — but in single-cell data the cell types
are usually what we hope to *find* after integration, a chicken-and-egg problem that motivates the local methods below.

✏️ **Your turn.** Shrinkage matters when batches are small. Take 15 random cells from each batch of `X_bal`, run the
location step of `combat_lite` with and without shrinkage (replace `g` by `g_hat`), and compare the corrected batch means
of a few genes. Which version is closer to the correction computed from all cells?

---
## 4 · Mutual nearest neighbours (MNN)

> 📍 **Where we are.** Linear methods estimate one correction per gene and batch from *all* cells. MNN (Haghverdi *et al.*
> 2018) estimates corrections **locally**, from pairs of cells that look like the same biological state in two batches.
> It needs no cell-type labels and tolerates types that exist in only one batch.

**The algorithm** (in PCA space, merging batches one at a time into a growing reference):

1. For every cell of the reference find its $k$ nearest cells in the new batch, and vice versa ([Module 08](08_knn_naive_bayes.ipynb)).
   A pair is **mutual** if each is among the other's $k$ nearest neighbours. Mutual pairs are taken to be the same cell
   state seen through two batches.
2. Each pair gives a **correction vector** (reference cell − new cell). Because the closest pairs lie at the facing edges
   of the two clouds, raw differences underestimate the shift (the "kissing" problem); like fastMNN, we instead use the
   difference between the *centres of mass* of each paired cell's neighbourhood in its own batch.
3. Every cell of the new batch is moved by a **Gaussian-kernel-weighted average** of the correction vectors of nearby pairs,
   $v(c)=\sum_p w_p(c)\,v_p\big/\sum_p w_p(c)$ with $w_p(c)=\exp(-\|z_c-z_{q(p)}\|^2/\sigma^2)$, so different regions of the
   data can be corrected differently.

**Assumptions.** (i) At least one cell population is shared between batches; (ii) the batch effect is small relative to
the differences between cell types (so that true pairs are mutual neighbours); (iii) the batch effect varies smoothly
over the data.

In [ ]:
# ── 4.1 MNN correction in PCA space, merging batches one by one ──
def mnn_correct(Z, batches, order, k=20, k_centre=20, sigma_scale=0.1):
    """Merge batches in `order` into a growing reference; returns corrected Z and the MNN pairs of each merge."""
    Zc, ref, pairs = Z.copy(), np.flatnonzero(batches == order[0]), []
    for b in order[1:]:
        new = np.flatnonzero(batches == b)
        R, Q = Zc[ref], Zc[new]
        nn_rq = NearestNeighbors(n_neighbors=k).fit(Q).kneighbors(R, return_distance=False)   # ref → new
        nn_qr = NearestNeighbors(n_neighbors=k).fit(R).kneighbors(Q, return_distance=False)   # new → ref
        A = sparse.csr_matrix((np.ones(nn_rq.size), (np.repeat(np.arange(len(R)), k), nn_rq.ravel())),
                              shape=(len(R), len(Q)))
        B = sparse.csr_matrix((np.ones(nn_qr.size), (nn_qr.ravel(), np.repeat(np.arange(len(Q)), k))),
                              shape=(len(R), len(Q)))
        pr, pq = A.multiply(B).nonzero()                                       # mutual pairs (ref index, new index)
        centre_r = R[NearestNeighbors(n_neighbors=k_centre).fit(R).kneighbors(R[pr], return_distance=False)].mean(1)
        centre_q = Q[NearestNeighbors(n_neighbors=k_centre).fit(Q).kneighbors(Q[pq], return_distance=False)].mean(1)
        vectors = centre_r - centre_q                                          # one correction vector per pair
        D = cdist(Q, Q[pq], "sqeuclidean")
        W = np.exp(-(D - D.min(axis=1, keepdims=True)) / (sigma_scale * np.median(D)))   # Gaussian kernel weights
        Zc[new] = Q + (W / W.sum(axis=1, keepdims=True)) @ vectors
        pairs.append((ref[pr], new[pq]))
        ref = np.concatenate([ref, new])                                       # the merged batch joins the reference
    return Zc, pairs

In [ ]:
# ── 4.2 MNN on the balanced design; 🧪 do the pairs link cells of the same true type? ──
Z_mnn, mnn_pairs = mnn_correct(P_bal, batch_bal, order=["batch1", "batch2", "batch3"])
for (r, q), b in zip(mnn_pairs, ["batch2", "batch3"]):
    print(f"merging {b}: {len(r):,} mutual pairs; both cells of the same true type in "
          f"{100 * np.mean(type_bal[r] == type_bal[q]):.1f} % of pairs")
emb_bal["MNN"] = Z_mnn
score_bal["MNN"] = integration_scores(Z_mnn, batch_bal, type_bal)
print(pd.DataFrame(score_bal).T.loc[["uncorrected", "ComBat-lite", "MNN"]].round(3).to_string())
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
keep = np.isin(batch_bal, ["batch1", "batch2"])
plot_embedding(P_bal[keep, :2], batch_bal[keep], ax=a1, title="MNN pairs between batch 1 and batch 2 (150 shown)")
r, q = mnn_pairs[0]
for i in np.random.default_rng(SEED + 4).choice(len(r), 150, replace=False):
    a1.plot(P_bal[[r[i], q[i]], 0], P_bal[[r[i], q[i]], 1], color="0.3", lw=0.3, alpha=0.6)
plot_embedding(pca(Z_mnn, 2)[0], type_bal, ax=a2, title="after MNN correction: coloured by true type")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Thousands of mutual pairs were found at each merge, and almost all of them join two cells of
the **same** true type (98.9 % and 97.9 %) — the ground-truth check of MNN's central assumption (the pair lines in the
left panel run between matching regions of the two batch clouds). Moving each cell by the smoothed correction vectors of
nearby pairs mixes the batches (mixing entropy 0.83, better than centring's 0.78) while keeping the types apart. MNN's
mixing is lower than ComBat-lite's 0.92 here — on a *global* batch shift a global method is hard to beat — but MNN makes
no assumption about composition, which will matter in Section 7.

⚠️ **When MNN fails.** If a cell type in the new batch has no counterpart in the reference, its nearest "mutual" partners
may be a *different* type, and the correction drags it onto the wrong population. The smaller the shared overlap, the
more fragile the pairs.

✏️ **Your turn.** Change the merge order to `["batch3", "batch1", "batch2"]` and `k` to 5 and to 50. How do the number of
pairs, their type purity and the final scores change?

### 📄 Research connection — From microarrays to single cells: batch correction

- **What the field does.** Batch effects were recognised as a pervasive problem in high-throughput genomics long before single-cell data
    (Leek *et al.* 2010). **ComBat** (Johnson, Li & Rabinovic 2007) introduced empirical-Bayes location/scale adjustment
    for microarrays and remains standard for bulk data. Single-cell data brought new methods that work locally and
    tolerate composition differences: **MNN** (Haghverdi *et al.* 2018), Seurat's **anchors** found in a shared CCA space
    (Stuart *et al.* 2019), **Scanorama** (Hie, Bryson & Berger 2019), **Harmony** (Korsunsky *et al.* 2019) and deep
    generative models such as **scVI** (Lopez *et al.* 2018).
- **What this notebook builds.** Per-gene centring and ComBat-lite (location/scale with the normal and inverse-gamma empirical-Bayes priors of
    ComBat, without its covariate model), MNN with centre-of-mass correction vectors and Gaussian smoothing in PCA space
    (close to the fastMNN variant), and below Harmony-lite and a batch-aware VAE. The production versions add
    cosine normalisation, batch-aware feature selection, careful merge orders and many engineering refinements.
- **Where the frontier is.** Open questions (as of 2025) include integrating across very different technologies (single-cell vs
    single-nucleus, RNA vs ATAC, different species), distinguishing technical batch from biological donor variation, and
    detecting over-correction without ground truth. Integrated embeddings are meant for visualisation, clustering and
    mapping; for differential expression most guidelines recommend modelling batch as a covariate on the raw counts
    (Module 24) rather than testing on corrected values.
- **Read next.** Johnson, Li & Rabinovic (2007) *Adjusting batch effects in microarray expression data using empirical Bayes
    methods*; Haghverdi *et al.* (2018) *Batch effects in single-cell RNA-sequencing data are corrected by matching mutual
    nearest neighbors*; Leek *et al.* (2010) *Tackling the widespread and critical impact of batch effects in
    high-throughput data*.

---
## 5 · Harmony-lite: soft clustering with a diversity penalty

> 📍 **Where we are.** Harmony (Korsunsky *et al.* 2019) is one of the most used integration methods because it is fast
> and works directly on PCA scores. It alternates two steps you already know: **soft k-means clustering**
> ([Modules 12–13](13_mixture_models_em.ipynb)) and **ridge regression** ([Module 07](07_ridge_lasso_sparse_regression.ipynb)) —
> with one twist that forces clusters to be shared by all batches.

**Step 1 — soft clustering that prefers diverse clusters.** Cells (on the unit sphere, i.e. cosine geometry) get soft
assignments $R_{ck}$ to $K$ centroids. Plain soft k-means would set $R_{ck}\propto\exp(-d_{ck}/\sigma)$, with $d_{ck}$ the
distance from cell $c$ to centroid $k$. Harmony multiplies by a **diversity penalty**:

$$R_{ck}\;\propto\;\exp\!\Big(-\frac{d_{ck}}{\sigma}\Big)\left(\frac{E_{kb(c)}+1}{O_{kb(c)}+1}\right)^{\theta},$$

*In words:* $O_{kb}$ is the (soft) number of cells from batch $b$ currently in cluster $k$ and $E_{kb}$ the number expected if
cluster $k$ had the overall batch proportions. If cluster $k$ already holds too many cells of this cell's batch
($O>E$), the factor is below 1 and the cell is pushed towards clusters where its batch is under-represented. $\theta$ sets
the strength (0 = plain soft k-means).

**Step 2 — per-cluster linear correction.** Within each cluster $k$, regress the PCA scores on the batch indicators with
cell weights $R_{ck}$ and a ridge penalty on the batch coefficients; then subtract each cell's batch effect, averaged over
its clusters: $z_c\leftarrow z_c-\sum_k R_{ck}\,\beta_{k,b(c)}$.

> 🔗 **Builds on [Module 13 §6](13_mixture_models_em.ipynb).** Soft k-means is EM for a mixture of spherical Gaussians with
> a shared width: the $R_{ck}$ are the E-step responsibilities and the centroid update is the M-step. Harmony adds to the
> clustering objective a term that rewards batch-diverse clusters, which appears as the multiplicative penalty above. Because the correction differs between clusters, different
cell types can be corrected differently — a *mixture of experts*. The two steps repeat until the assignments stabilise.
Our version simplifies Harmony's objective and convergence checks but keeps both mechanisms.

In [ ]:
# ── 5.1 Harmony-lite ──
def harmony_lite(Z, batches, n_clusters=20, theta=2.0, sigma=0.1, n_rounds=10, ridge=1.0, seed=SEED):
    """Alternate diversity-penalised soft k-means (cosine geometry) and per-cluster ridge correction of PCA scores."""
    levels = np.unique(batches)
    Phi = (batches[:, None] == levels).astype(float)              # cells × batches indicator matrix
    design = np.hstack([np.ones((len(Z), 1)), Phi])               # intercept + batch indicators
    penalty = ridge * np.eye(design.shape[1]); penalty[0, 0] = 0  # do not shrink the intercept
    Zc, Y = Z.copy(), None
    for _ in range(n_rounds):
        Zn = Zc / np.linalg.norm(Zc, axis=1, keepdims=True)
        if Y is None:                                             # initialise centroids with k-means
            Y = KMeans(n_clusters, n_init=3, random_state=seed).fit(Zn).cluster_centers_
            Y /= np.linalg.norm(Y, axis=1, keepdims=True)
        for _ in range(5):                                        # Step 1: penalised soft clustering
            dist = 2 * (1 - Zn @ Y.T)                              # cosine distance to each centroid
            R = np.exp(-(dist - dist.min(axis=1, keepdims=True)) / sigma)
            R /= R.sum(axis=1, keepdims=True)
            O, E = R.T @ Phi, R.sum(axis=0)[:, None] * Phi.mean(axis=0)[None, :]
            R *= (((E + 1) / (O + 1)) ** theta @ Phi.T).T          # diversity penalty for each cell's own batch
            R /= R.sum(axis=1, keepdims=True)
            Y = R.T @ Zn
            Y /= np.linalg.norm(Y, axis=1, keepdims=True)
        Zc = Z.copy()                                              # Step 2: correct the ORIGINAL scores
        for k in range(n_clusters):
            Dw = design * R[:, [k]]
            beta = np.linalg.solve(Dw.T @ design + penalty, Dw.T @ Z)
            beta[0] = 0                                            # keep the cluster's own centre
            Zc -= Dw @ beta
    return Zc, R

In [ ]:
# ── 5.2 Harmony-lite on the balanced design, with and without the diversity penalty ──
for theta, name in [(0.0, "Harmony-lite θ=0"), (2.0, "Harmony-lite")]:
    Z_h, R_h = harmony_lite(P_bal, batch_bal, theta=theta)
    hard = R_h.argmax(axis=1)
    n_shared = sum(len(np.unique(batch_bal[hard == k])) == 3 for k in np.unique(hard))
    print(f"θ = {theta}: {n_shared} of {len(np.unique(hard))} final clusters contain cells from all 3 batches")
    score_bal[name] = integration_scores(Z_h, batch_bal, type_bal)
    if theta > 0:
        emb_bal["Harmony-lite"] = Z_h
print(pd.DataFrame(score_bal).T.loc[["uncorrected", "Harmony-lite θ=0", "Harmony-lite"]].round(3).to_string())

🔍 **What just happened?** Without the penalty (θ = 0), soft k-means simply finds the batch islands: **none** of the 20
final clusters contains cells from all three batches, so a regression on batch *within* a cluster has nothing to remove,
and the mixing entropy stays at 0.06. With θ = 2, **all** 20 clusters contain all three batches; the per-cluster regression
then removes the batch differences inside each of them, and mixing rises to 0.82 with biology conserved (ARI 0.998). The
diversity penalty is what makes Harmony work.

⚠️ **The diversity penalty assumes every cluster should contain every batch** — a "balanced composition" assumption again,
but local instead of global. When a cell type truly exists in only some batches, the penalty pushes cells of other types
into its cluster. Section 7 tests exactly this.

✏️ **Your turn.** Run `harmony_lite` with `theta=5` and with `n_clusters=6` on the balanced data. Does more penalty always
mean better mixing, and what happens to the bio score?

---
## 6 · A batch-conditioned VAE (the scVI idea)

> 📍 **Where we are.** [Module 25 §7](25_variational_inference_vae.ipynb) trained a negative-binomial VAE whose encoder
> and decoder receive the batch label. Here we use a compact copy of it as an integration method and score it like the
> others. Two small differences from Module 25: this version always receives a batch vector (no batch-free option, no
> Gaussian decoder), and the gene-wise dispersion parameter is called `log_theta` instead of `log_disp`.

**Recap.** Each cell has a latent state $z\sim\mathcal N(0,I)$. The decoder turns $(z,\text{batch})$ into gene proportions
$\rho(z,b)$, and the raw counts are negative binomial with mean $\ell_c\,\rho_g(z_c,b_c)$, where $\ell_c$ is the observed
library size. The encoder outputs $q(z\mid x,b)$. Training maximises the ELBO, "reconstruction − KL". Because the decoder
can explain batch differences from the batch label it is given, and the KL term penalises any information in $z$ that the
decoder does not need, $z$ is pushed towards a batch-free description of the cell. Unlike the linear methods, the decoder
is non-linear, so batch effects may differ between cell types; unlike MNN and Harmony, it models raw counts.

In [ ]:
# ── 6.1 The batch-aware NB VAE of Module 25 (compact copy, part 1: likelihood and model) ──
def nb_log_prob(x, mu, theta, eps=1e-8):
    """log NB(x | mean mu, inverse dispersion theta), elementwise on tensors."""
    log_theta_mu = torch.log(theta + mu + eps)
    return (torch.lgamma(x + theta) - torch.lgamma(theta) - torch.lgamma(x + 1)
            + theta * (torch.log(theta + eps) - log_theta_mu) + x * (torch.log(mu + eps) - log_theta_mu))


class MiniVAE(nn.Module):
    """NB VAE: encoder q(z | x, batch), decoder proportions ρ(z, batch), counts ~ NB(ℓ·ρ, θ_g)."""

    def __init__(self, n_genes, n_latent=10, n_hidden=128, n_batches=0):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_genes + n_batches, n_hidden), nn.ReLU())
        self.enc_mu, self.enc_logvar = nn.Linear(n_hidden, n_latent), nn.Linear(n_hidden, n_latent)
        self.decoder = nn.Sequential(nn.Linear(n_latent + n_batches, n_hidden), nn.ReLU(), nn.Linear(n_hidden, n_genes))
        self.log_theta = nn.Parameter(torch.zeros(n_genes))

    def encode(self, x_in, b):
        h = self.encoder(torch.cat([x_in, b], dim=1))
        return self.enc_mu(h), self.enc_logvar(h)

    def elbo_terms(self, x_in, x_obs, lib, b):
        mu_z, logvar_z = self.encode(x_in, b)
        z = mu_z + torch.exp(0.5 * logvar_z) * torch.randn_like(mu_z)          # reparameterisation trick
        rho = torch.softmax(self.decoder(torch.cat([z, b], dim=1)), dim=1)
        rec = nb_log_prob(x_obs, lib * rho, torch.exp(self.log_theta)).sum(dim=1)
        kl = 0.5 * (mu_z ** 2 + torch.exp(logvar_z) - 1 - logvar_z).sum(dim=1)
        return rec, kl

In [ ]:
# ── 6.2 Training with KL warm-up (part 2) and the batch VAE on the balanced design ──
def train_batch_vae(X, C, batches, epochs=80, batch_size=256, lr=1e-3, warmup=10, seed=SEED):
    """Train a batch-conditioned NB VAE; returns the model and the posterior means (the integrated embedding)."""
    torch.manual_seed(seed)
    levels = np.unique(batches)
    B = torch.as_tensor((batches[:, None] == levels).astype(np.float32))
    x_in, x_obs = torch.as_tensor(X), torch.as_tensor(C)
    lib = x_obs.sum(dim=1, keepdim=True)
    model = MiniVAE(X.shape[1], n_batches=len(levels))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        w = min(1.0, (epoch + 1) / warmup)                       # KL weight rises from 0 to 1
        perm = torch.randperm(len(X))
        for start in range(0, len(X), batch_size):
            i = perm[start:start + batch_size]
            rec, kl = model.elbo_terms(x_in[i], x_obs[i], lib[i], B[i])
            loss = -(rec - w * kl).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return model, model.encode(x_in, B)[0].numpy()


t0 = time.time()
vae_bal, Z_vae = train_batch_vae(X_bal, C_bal, batch_bal)
print(f"batch VAE trained in {time.time() - t0:.0f} s (80 epochs × 16 mini-batches)")
emb_bal["batch VAE"] = Z_vae
score_bal["batch VAE"] = integration_scores(Z_vae, batch_bal, type_bal)
print(pd.DataFrame(score_bal).T.loc[["uncorrected", "ComBat-lite", "MNN", "Harmony-lite", "batch VAE"]].round(3).to_string())

🔍 **What just happened?** The batch-conditioned VAE mixes the batches (mixing entropy 0.86, second only to ComBat-lite)
and gives the most compact cell types of all methods (rescaled silhouette 0.73 against about 0.63 for the others; the
highest bio score, 0.93), with ARI, NMI and k-NN accuracy close to 1 — without any linear-shift assumption and working
from raw counts. It is also the slowest method here, and the only one with many training choices (epochs, latent
size, KL schedule) that change its result.

⚠️ **Train long enough.** VAE integration improves with training: an under-trained model still carries batch
information in $z$. (Try it in the ✏️ below.) Production tools use early stopping on a validation ELBO rather than a
fixed number of epochs.

✏️ **Your turn.** Re-train with `epochs=30` and compare the mixing entropy with the 80-epoch model. Then try giving the
batch label only to the decoder (pass a zero vector to the encoder): does mixing change?

---
## 7 · 🧪 Benchmark: mixing versus conservation in two designs

> 📍 **Where we are.** Every method has been scored on the balanced design. Now we rerun all of them on the
> **missing-type** design of Section 3.3 (no B cells in batch 2) and compare, as the scIB benchmark does on real data.

We add one targeted ground-truth check for over-correction: since batch 2 contains **no** B cells, every batch-2 cell
whose neighbours vote "B" is an artefact of the integration — biology from other batches painted onto it.

In [ ]:
# ── 7.1 All methods on the missing-type design ──
t0 = time.time()
emb_mis = {"uncorrected": P_mis,
           "centring": pca(versions["centring"], 20)[0],
           "ComBat-lite": pca(versions["ComBat-lite"], 20)[0],
           "MNN": mnn_correct(P_mis, batch_mis, order=["batch1", "batch2", "batch3"])[0],
           "Harmony-lite": harmony_lite(P_mis, batch_mis)[0],
           "batch VAE": train_batch_vae(X_mis, C_mis, batch_mis)[1]}
print(f"six embeddings computed in {time.time() - t0:.0f} s")


def called_b_in_batch2(Z, batches, types, k=30):
    """Fraction of batch-2 cells (which are never B cells) whose k nearest neighbours vote 'B'."""
    idx = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z[batches == "batch2"], return_distance=False)[:, 1:]
    return np.mean((types[idx] == "B").mean(axis=1) > 0.5)


score_mis = {}
for name, Z in emb_mis.items():
    score_mis[name] = integration_scores(Z, batch_mis, type_mis)
    score_mis[name]["batch-2 cells called B"] = called_b_in_batch2(Z, batch_mis, type_mis)
cols = ["batch score", "bio score", "overall"]
bench = pd.concat({"balanced": pd.DataFrame(score_bal).T.loc[list(emb_mis), cols],
                   "missing B in batch 2": pd.DataFrame(score_mis).T[cols + ["batch-2 cells called B", "ARI"]]}, axis=1)
print(bench.round(3).to_string())

In [ ]:
# ── 7.2 The trade-off at a glance ──
fig, ax = plt.subplots(figsize=(7, 4.2))
colors = dict(zip(emb_mis, ["0.5", "tab:brown", "tab:orange", "tab:green", "tab:purple", "tab:blue"]))
for name in emb_mis:
    b0, s0 = score_bal[name]["batch score"], score_bal[name]["bio score"]
    b1, s1 = score_mis[name]["batch score"], score_mis[name]["bio score"]
    ax.annotate("", xy=(b1, s1), xytext=(b0, s0), arrowprops=dict(arrowstyle="->", color=colors[name], lw=1))
    ax.scatter([b0], [s0], color=colors[name], s=40, marker="o", label=name)
    ax.scatter([b1], [s1], color=colors[name], s=40, marker="s")
ax.set_xlabel("batch score (mixing)  →  better"); ax.set_ylabel("bio score (conservation)  →  better")
ax.set_title("Circles: balanced design; squares: B cells missing from batch 2 (SIMULATED)")
ax.legend(fontsize=7, loc="lower center")
plt.tight_layout(); plt.show()
show_embeddings({k: emb_mis[k] for k in ["uncorrected", "ComBat-lite", "MNN", "Harmony-lite", "batch VAE"]},
                batch_mis, type_mis, "SIMULATED missing-type design (no B cells in batch 2)")

🔍 **What just happened?** Read the table and the arrows from circle (balanced) to square (missing type) for each method.

- **Uncorrected** data keep biology locally but do not mix at all — the baseline every method must beat.
- **Centring** loses biology when B cells are missing (bio score 0.91 → 0.81, ARI 0.75); **ComBat-lite** holds up better
  (bio 0.89, ARI 0.97) but mixes less (batch score 0.80 → 0.62). Both carry the spurious B-marker signature of Section 3.3,
  which none of these summary metrics measures. **Summary metrics can miss biologically important distortions** — targeted
  checks are needed.
- **MNN** copes well with the missing type: its mixing even rises (0.62 → 0.78; the two designs are different simulated
  data sets), its bio score dips slightly (0.87), and almost no batch-2 cell is called B (0.2 %).
- **Harmony-lite** reaches the highest mixing (0.88) and the **highest overall score** (0.875) in the missing-type design —
  yet 6.4 % of batch-2 cells, none of which is a B cell, now have mostly B-cell neighbours: the diversity penalty insists
  that the B cluster should contain batch-2 cells too, and pulls other cells in. The overall score rewards a method that
  has created an artefact.
- **The batch VAE** conserves biology best in both designs (bio 0.93; ARI 0.996 with the missing type), mixes moderately
  (batch score 0.70) and calls no batch-2 cell B.

There is no universal winner. Which method is "best" depends on the design (balanced or not), on the kind of batch effect
(our simulator's is global, which favours linear methods), and on how mixing and conservation are weighted — the
conclusion of large benchmarks on real data as well (Tran *et al.* 2020; Luecken *et al.* 2022).

✏️ **Your turn.** Change the overall-score weights to 0.6 × batch + 0.4 × bio. Does the ranking of methods change?
Which method would you choose for an atlas in which some cell types are expected to be donor-specific?

### 📄 Research connection — Benchmarking integration

- **What the field does.** As integration methods multiplied, so did benchmarks. **kBET** (Büttner *et al.* 2019) introduced a
    neighbourhood test of batch mixing; Tran *et al.* (2020) compared 14 methods on simulated and real data with mixing and
    conservation metrics; **scIB** (Luecken *et al.* 2022) benchmarked 16 methods on 13 integration tasks with 14 metrics
    grouped into batch removal and bio-conservation, and found that the ranking depends on the task, with scANVI,
    Scanorama, scVI and scGen performing well particularly on complex atlas-level tasks.
- **What this notebook builds.** A compact scIB-style scoreboard on simulated data with known truth: a type-aware neighbourhood entropy and a
    kBET-like test for mixing; ARI, NMI, silhouette and k-NN accuracy for conservation; the 0.4/0.6 weighting; and a
    targeted over-correction check. Simulation lets us use the true types, which real benchmarks must replace with expert
    annotations.
- **Where the frontier is.** Metrics themselves are debated: several scores can be "gamed" (Section 2's random embedding), label-based
    scores inherit annotation errors, and biologically meaningful variation *within* cell types (states, gradients) is hard
    to score. Newer benchmarks add metrics for preserving intra-type structure and for downstream tasks such as reference
    mapping and differential expression.
- **Read next.** Luecken *et al.* (2022) *Benchmarking atlas-level data integration in single-cell genomics*; Tran *et al.*
    (2020) *A benchmark of batch-effect correction methods for single-cell RNA sequencing data*; Büttner *et al.* (2019)
    *A test metric for assessing single-cell RNA-seq batch correction*.

---
## 8 · Reference mapping and label transfer

> 📍 **Where we are.** Building an atlas is expensive and done once; using it is done again and again. **Reference
> mapping** places the cells of a new *query* sample into the reference's coordinate system **without refitting the
> reference**, and **label transfer** borrows the reference's cell-type annotations ([Module 08](08_knn_naive_bayes.ipynb)
> k-NN classification). The hard part is to notice query cells that the reference does not contain.

**Our set-up (balanced design).** The *reference* = batches 1 and 2, from which we delete all dendritic cells (pretend the
atlas never captured them). The *query* = batch 3, which contains all six types — so its dendritic cells are a "novel"
population. We map in the simplest way: fit PCA on the reference only and **project** the query with the reference's mean
and loadings. Each query cell gets the majority label of its $k=15$ nearest reference cells and a **confidence** = the
fraction of those neighbours that agree; we also record the mean distance to them.

> 🔗 **Builds on [Module 14 §4](14_pca_dimensionality_reduction.ipynb) and [Module 08](08_knn_naive_bayes.ipynb).**
> Projecting new samples with the *training* mean and loadings is exactly how you projected test data in Module 14
> ("always centre with the training mean"); label transfer is a k-NN classifier whose training set is the reference.

In [ ]:
# ── 8.1 Reference (batches 1–2, no dendritic cells) and query (batch 3): project, then transfer labels ──
held_out = "Dendritic"
is_ref = (batch_bal != "batch3") & (type_bal != held_out)
is_qry = batch_bal == "batch3"
ref_mean = X_bal[is_ref].mean(axis=0)
_, ref_loadings, _ = pca(X_bal[is_ref], 20)
Z_ref = (X_bal[is_ref] - ref_mean) @ ref_loadings.T
Z_qry = (X_bal[is_qry] - ref_mean) @ ref_loadings.T             # the query is projected, never refitted


def transfer_labels(Z_reference, labels_reference, Z_query, k=15):
    """k-NN label transfer: majority label, confidence (fraction of agreeing neighbours), mean neighbour distance."""
    dist, idx = NearestNeighbors(n_neighbors=k).fit(Z_reference).kneighbors(Z_query)
    lev = np.unique(labels_reference)
    votes = np.stack([(labels_reference[idx] == l).sum(axis=1) for l in lev], axis=1)
    return lev[votes.argmax(axis=1)], votes.max(axis=1) / k, dist.mean(axis=1)


pred, conf, ndist = transfer_labels(Z_ref, type_bal[is_ref], Z_qry)
truth_q = type_bal[is_qry]
known = truth_q != held_out
print(f"reference: {is_ref.sum()} cells, query: {is_qry.sum()} cells ({(~known).sum()} of them {held_out}, a type "
      f"absent from the reference)")
print(f"🧪 label-transfer accuracy on query cells of reference types: {np.mean(pred[known] == truth_q[known]):.3f}")
print("per type:", {str(t): round(float(np.mean(pred[truth_q == t] == t)), 3) for t in np.unique(truth_q[known])})
print(f"the {held_out} cells were labelled as:", pd.Series(pred[~known]).value_counts().to_dict())

🔍 **What just happened?** Even though the query comes from a batch the reference never saw, k-NN label transfer after
projection is very accurate for the five reference types (99.8 % overall, at least 99.5 % per type). In this simulation the batch shift is
smaller than the differences between cell types, so a query cell's nearest reference cells are still of its own type —
the batch effect moves the query cloud, but not across type boundaries. The 211 dendritic cells, which the reference does not
contain, are **all** given some reference label (CD4 T, CD8 T or monocyte) — a k-NN classifier cannot say "none of the
above". Whether we can flag
them depends on the confidence score.

In [ ]:
# ── 8.2 🧪 Can we detect the query-only cell type? Confidence and distance as "unknown" scores ──
is_novel = ~known
auc_conf = roc_auc_score(is_novel, 1 - conf)                    # low confidence → more likely novel
auc_dist = roc_auc_score(is_novel, ndist)                       # far from the reference → more likely novel
print(f"AUROC for flagging {held_out} cells: 1 − confidence = {auc_conf:.3f} | distance to reference = {auc_dist:.3f}")
flag = conf < 0.7                                               # a simple rule: call 'unknown' below 70 % agreement
print(f"rule 'unknown if confidence < 0.7': flags {100 * flag[is_novel].mean():.0f} % of {held_out} cells "
      f"and {100 * flag[known].mean():.1f} % of known-type cells")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
bins = np.linspace(0, 1, 16)
a1.hist(conf[known], bins=bins, density=True, alpha=0.6, color="tab:blue", label="reference types")
a1.hist(conf[is_novel], bins=bins, density=True, alpha=0.6, color="tab:red", label=f"{held_out} (novel)")
a1.axvline(0.7, color="k", lw=0.8); a1.set_xlabel("label-transfer confidence"); a1.set_ylabel("density")
a1.set_title("Novel cells get lower confidence"); a1.legend(fontsize=7)
a2.hist(ndist[known], bins=30, density=True, alpha=0.6, color="tab:blue", label="reference types")
a2.hist(ndist[is_novel], bins=30, density=True, alpha=0.6, color="tab:red", label=f"{held_out} (novel)")
a2.set_xlabel("mean distance to the 15 nearest reference cells"); a2.set_title("…and sit farther from the reference")
a2.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The confidence score separates the novel dendritic cells from known types well (AUROC 0.97;
1.0 would be perfect, 0.5 chance): their 15 nearest reference cells disagree about what they are, because no reference
population matches them, whereas almost every known-type cell has unanimous neighbours. The simple rule "unknown below
70 % agreement" flags 68 % of the dendritic cells and only 1.6 % of the known-type cells. The distance score is a second,
weaker signal (AUROC 0.83) — novel cells tend to lie far from every reference cell — but the batch shift also moves *all*
query cells away from the reference, which blurs distance-based novelty detection.

Now the tempting alternative: instead of projecting, **integrate** reference and query together (Harmony-lite on their
joint PCA) and transfer labels in the integrated space.

In [ ]:
# ── 8.3 Integrate-then-transfer: does forced mixing hide the novel cells? ──
sel = is_ref | is_qry
Z_joint, _ = harmony_lite(pca(X_bal[sel], 20)[0], batch_bal[sel])
in_ref = is_ref[sel]
pred_h, conf_h, ndist_h = transfer_labels(Z_joint[in_ref], type_bal[is_ref], Z_joint[~in_ref])
summary = pd.DataFrame({
    "projection onto reference PCA": [np.mean(pred[known] == truth_q[known]), auc_conf, auc_dist,
                                      np.mean(conf[is_novel])],
    "Harmony-lite on reference + query": [np.mean(pred_h[known] == truth_q[known]),
                                          roc_auc_score(is_novel, 1 - conf_h), roc_auc_score(is_novel, ndist_h),
                                          np.mean(conf_h[is_novel])]},
    index=["accuracy on reference types", "AUROC novel (1 − confidence)", "AUROC novel (distance)",
           f"mean confidence of {held_out} cells"])
print(summary.round(3).to_string())
print(f"after integration the {held_out} cells were labelled as:", pd.Series(pred_h[is_novel]).value_counts().to_dict())

🔍 **What just happened?** Joint integration transfers labels just as accurately for the known types (99.7 %), but look
at the novelty rows: after integration **all 211 dendritic cells are labelled "Monocyte" with an average confidence of
0.998**, and both novelty scores fall to chance level (AUROC 0.48 for confidence, 0.57 for distance). Harmony's diversity penalty wants every cluster to contain cells from every
batch, so the query-only population is pulled towards reference clusters — exactly the over-integration seen in
Section 7. **An integration that is good at mixing can be bad at discovery.** This is why reference-mapping tools
(Azimuth, scArches, scANVI) keep the reference fixed, map the query into it, and report per-cell uncertainty.

⚠️ **Projection works here because our batch effect is a shift shared by all types and smaller than the type
differences.** With stronger or cell-type-specific batch effects, naive projection mislabels cells; then the query must be
aligned first — e.g. by fine-tuning only the batch-specific weights of a reference VAE ("architectural surgery", scArches)
— while keeping the reference fixed.

✏️ **Your turn.** Hold out `"B"` instead of `"Dendritic"`. B cells are lymphoid like T and NK cells: are they harder or
easier to flag as novel than dendritic cells, and which reference type absorbs them?

### 📄 Research connection — Reference atlases and mapping

- **What the field does.** Large references now exist for many tissues (e.g. through the Human Cell Atlas and CELLxGENE), and mapping
    tools let any lab use them. Seurat's reference mapping and **Azimuth** project queries onto a fixed multimodal
    reference (Hao *et al.* 2021); **scArches** (Lotfollahi *et al.* 2022) maps queries onto a pre-trained deep generative
    reference (scVI, scANVI, trVAE) by *architectural surgery* — adding new batch-specific input weights and fine-tuning only
    those, so the reference is never retrained and never needs to be shared as raw data; **scANVI** (Xu *et al.* 2021)
    adds cell-type labels to the VAE for semi-supervised annotation and transfer with probabilities.
- **What this notebook builds.** PCA projection of a query onto a fixed reference, k-NN label transfer with a confidence score, ground-truth
    evaluation of accuracy and of novel-type detection (AUROC), and a demonstration that integrating query and reference
    together can mask a query-only population.
- **Where the frontier is.** Calibrated uncertainty for label transfer, detection of truly novel or disease-specific states (rather than
    forcing them into healthy-reference labels), mapping across technologies and species, and the role of large pre-trained
    "foundation" models of single cells as references (Module 43) are active research areas as of 2025.
- **Read next.** Lotfollahi *et al.* (2022) *Mapping single-cell data to reference atlases by transfer learning* (scArches);
    Hao *et al.* (2021) *Integrated analysis of multimodal single-cell data* (Seurat v4, Azimuth); Xu *et al.* (2021)
    (scANVI).

---
## 9 · 🧭 From toy to real data

> 📍 **Where we are.** Everything above ran on 4,000 simulated cells with one global batch effect. This section is
> reading: what changes with real atlases.

**Scale and compute.** Atlas projects integrate $10^5$–$10^7$ cells from tens to hundreds of samples. Harmony and
PCA-based methods run on a laptop for $10^5$–$10^6$ cells; scVI/scANVI train on a GPU in minutes to hours; reference
mapping of a new sample takes minutes.

**Public data and resources.** [CELLxGENE](https://cellxgene.cziscience.com/) (curated data sets and the Census API),
the [Human Cell Atlas](https://www.humancellatlas.org/) data portal, [Azimuth](https://azimuth.hubmapconsortium.org/)
references for many human tissues, and the integration tasks of the [scIB benchmark](https://github.com/theislab/scib).
PBMC experiments such as Kang *et al.* (2018), with donors multiplexed across conditions, are classic test cases.

**Production tools.** Harmony (R and Python), [scvi-tools](https://scvi-tools.org/) (scVI, scANVI, and scArches-style
query mapping), [Seurat](https://satijalab.org/seurat/) (anchor-based integration, reference mapping), Scanorama, BBKNN,
fastMNN (Bioconductor `batchelor`) and Scanpy's `pp.combat`. Most accept an AnnData or Seurat object plus a `batch_key`.

**What changes with real data — the first three things that go wrong.**

1. **Choosing what "batch" means.** Real studies nest technical batches, donors, chemistries and labs, and some of them are
   partly confounded with biology (all patients from one hospital). Integrating over *donor* removes donor-to-donor biology
   (genotype, age, sex); integrating over the wrong key, or over a key confounded with disease, erases the signal you want
   (Section 1.3). Tabulate the design first.
2. **Cell-type-specific batch effects and unequal composition.** Real batch effects differ between cell types (e.g.
   dissociation stress hits some types harder), and many studies have types present in only some samples. Global linear
   corrections then fail, and strongly mixing methods over-correct (Sections 3.3 and 7). Check corrected data with
   targeted, biology-based tests (known markers, known type-specific populations), not just summary metrics.
3. **Using corrected data for the wrong task.** Integrated embeddings are for visualisation, clustering and mapping. For
   differential expression, use raw counts with batch (and donor) in the model, ideally with pseudobulk per donor
   ([Module 24](24_glm_count_data.ipynb)); corrected values can create or remove differences.

---
## 📌 Key takeaways

- **Batch effects** can dominate the largest axes of variation (here PC1–PC2), so naive analyses find processing batches
  instead of cell types.
- **A confounded design is unfixable**: when batch and condition coincide the design matrix is rank-deficient, and
  removing batch removes the biology (the interferon response went to exactly 0). Only good experimental design helps.
- **Judge integration on two axes** — batch mixing *and* biology conservation — with ground truth where possible; a random
  embedding mixes perfectly. Summary metrics can miss targeted distortions, so add biology-based checks.
- **Linear methods** (centring, ComBat with empirical-Bayes shrinkage) are right when the batch effect is a global shift
  and composition is balanced; with a missing cell type they **over-correct** (a spurious B-cell signature in batch 2).
- **Local methods**: MNN corrects from mutual pairs (98–99 % of pairs linked the same true type); Harmony alternates
  diversity-penalised soft clustering and per-cluster regression — and its penalty over-mixes when a type is missing
  (6 % of batch-2 cells pulled into the B-cell cluster) while still earning the best overall score.
- **A batch-conditioned VAE** (scVI idea) models raw counts and non-linear batch effects; it needs enough training and has
  more tuning choices.
- **Reference mapping**: project the query onto a fixed reference, transfer labels by k-NN with a confidence score, and
  flag low-confidence cells (novelty AUROC 0.97); **integrating query and reference together hid the novel cell type**
  (all of it labelled "Monocyte", AUROC 0.48).

---
## ✅ Checkpoint

1. A study processes all 4 patients on Monday and all 4 controls on Tuesday. Write the design matrix with intercept, 'Tuesday' and 'control' columns. Why can no batch-correction method recover the disease effect?
   <details><summary>Answer</summary>The 'Tuesday' and 'control' columns are identical, so the matrix has rank 2, not 3: any split of the observed difference between day and disease fits equally well (non-identifiable). Removing the day effect removes the disease effect, keeping one keeps the other — as in Section 1.3, where centring set the interferon response to 0.</details>
2. Why does a random embedding score well on batch-mixing metrics, and what does that imply for benchmarking?
   <details><summary>Answer</summary>Random coordinates make every neighbourhood a random sample of all cells, so batches look perfectly mixed. But the embedding contains no biology (ARI ≈ 0). Integration must therefore be scored on mixing *and* conservation together.</details>
3. Explain in two sentences how batch-mean centring creates a B-cell signature in batch 2 when batch 2 has no B cells.
   <details><summary>Answer</summary>Batch 2's mean expression of B-cell markers is lower because it lacks B cells — a biological difference. Centring subtracts each batch's mean, so it adds that difference back to every batch-2 cell, giving T, NK and myeloid cells spurious B-marker expression.</details>
4. In ComBat's posterior mean $\gamma^*=\frac{n\tau^2\hat\gamma+\delta^2\bar\gamma}{n\tau^2+\delta^2}$, compute $\gamma^*$ for $\hat\gamma=1.0$, $\bar\gamma=0.2$, $\tau^2=0.1$, $\delta^2=1$ with $n=5$ and with $n=1000$ cells.
   <details><summary>Answer</summary>$n=5$: $(0.5\cdot1.0+1\cdot0.2)/(0.5+1)=0.47$ — strongly shrunk towards 0.2. $n=1000$: $(100+0.2)/(100+1)\approx0.99$ — essentially the gene's own estimate. Shrinkage matters for small batches.</details>
5. What does MNN assume, and what goes wrong if a cell type in the new batch has no counterpart in the reference?
   <details><summary>Answer</summary>That some populations are shared, that the batch effect is smaller than type differences (so true pairs are mutual neighbours) and that it varies smoothly. A type without counterpart can form mutual pairs with a *different* type and be dragged onto it by the correction.</details>
6. Code reading: in `harmony_lite`, what does the line `R *= (((E + 1) / (O + 1)) ** theta @ Phi.T).T` do, and what happens with `theta=0`?
   <details><summary>Answer</summary>For each cell it multiplies its soft assignment to cluster $k$ by $((E_{kb}+1)/(O_{kb}+1))^\theta$ for its own batch $b$: clusters that already contain too many cells of that batch become less attractive (diversity penalty). With $\theta=0$ the factor is 1 and the step is plain soft k-means.</details>
7. Why is the confidence of k-NN label transfer a reasonable novelty score, and why can integrating query and reference jointly make novel cells harder to detect?
   <details><summary>Answer</summary>A cell of a type absent from the reference has neighbours from several reference types that disagree, so its agreement fraction is low. Joint integration actively pulls query cells towards reference cells (Harmony's diversity penalty), so a query-only population is merged into reference clusters and receives confident but wrong labels.</details>
8. Why do guidelines recommend differential expression on raw counts with batch as a covariate rather than on batch-corrected values?
   <details><summary>Answer</summary>Corrected values are outputs of a model fitted to all cells; they can remove real differences (over-correction) or create artefacts, and their noise is no longer described by a count model, so tests are miscalibrated. A count GLM with batch (and donor) terms — e.g. pseudobulk per donor (Module 24) — estimates the effect while accounting for batch.</details>

## 🧠 Questions & discussion

1. A consortium integrates healthy and diseased lung samples from 40 donors. After integration, a disease-associated macrophage state that was obvious in each diseased sample has disappeared from the atlas. List possible causes and the checks that would distinguish them.
2. When is the *donor* a batch to be removed and when is donor-to-donor variation the biology of interest (e.g. genetics of gene expression, ageing)? How would you design the integration in each case?
3. Section 7 found no universal winner. Which properties of your own data (number of batches, composition differences, technology differences, size) would make you choose a linear method, MNN, Harmony or a VAE?
4. Reference atlases are built mostly from donors of a few ancestries and from adult tissue. What could go wrong when cells from an under-represented population or from a child are mapped onto them, and how would confidence scores help or fail?
5. Batch correction can make figures look cleaner. What would you require in a methods section to be convinced that an integration did not remove the biological effect a paper reports?

## 🚀 Bonus — going further

- **Harmony on the VAE.** Run `harmony_lite` on the latent space of the batch VAE (instead of PCA scores). Does stacking two methods improve the scores, or over-correct?
- **Batch-aware gene selection.** Select highly variable genes within each batch and keep genes variable in at least two batches. Repeat the benchmark: does it change the mixing of the uncorrected and corrected data?
- **Cell-type-specific batch effects.** Write a small wrapper that multiplies the counts of batch-3 monocytes for 50 genes by a random factor (e.g. resample with `sample_negative_binomial` at the modified mean). Which methods cope, and how does ComBat-lite fare?
- **Partial label transfer and scArches-style surgery.** Train the batch VAE on the reference only (with a third, unused batch slot), freeze all weights except those attached to the batch-3 input, fine-tune them on the query, and transfer labels in the latent space. Compare accuracy and novelty AUROC with the PCA projection of Section 8.
- **Real data.** In a fresh Colab, install `scanpy`, `harmonypy` and `scvi-tools`, load a public multi-batch PBMC data set from CELLxGENE, and run Harmony and scVI with `batch_key` set; evaluate with the scIB metrics package.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 31 · Batch effects, data integration and reference mapping
Biological question: how can technical batch effects be removed without erasing biology, and how can a new
                     sample be mapped onto a reference atlas while noticing novel cells?
Data (simulated): simulate_scrna PBMC-like counts, 6 donors in 3 batches; designs: balanced, missing type
                  (no B cells in batch 2), confounded (batch = condition); truth: types, batches, true IFN effect
Method: per-gene centring; ComBat-lite (EB location/scale); MNN (centre-of-mass vectors, Gaussian smoothing);
        Harmony-lite (diversity-penalised soft k-means + per-cluster ridge); batch-conditioned NB VAE;
        PCA projection + k-NN label transfer with confidence
Key equation: γ* = (n τ² γ̂ + δ*² γ̄)/(n τ² + δ*²);  R_ck ∝ exp(−d_ck/σ)·((E_kb+1)/(O_kb+1))^θ;
              overall = 0.4·batch score + 0.6·bio score
Key parameters: k (neighbours), σ (kernel width), θ (diversity), number of clusters, ridge λ, VAE epochs
Ground-truth check result: (scores per method and design; spurious B-marker excess in batch 2; MNN pair purity;
                           batch-2 cells called B; label-transfer accuracy; novelty AUROC)
What changed when I modified … (θ / k / design / held-out type): …
One failure mode: (e.g. batch centring with a missing type painted a B-cell signature onto batch 2)
Connection to research (paper/tool): ComBat, MNN, Harmony, scVI, scIB, scArches/Azimuth
```

## 📚 References

- Johnson W. E., Li C., Rabinovic A. (2007) Adjusting batch effects in microarray expression data using empirical Bayes methods. *Biostatistics* 8:118–127.
- Leek J. T. *et al.* (2010) Tackling the widespread and critical impact of batch effects in high-throughput data. *Nat Rev Genet* 11:733–739.
- Haghverdi L., Lun A. T. L., Morgan M. D., Marioni J. C. (2018) Batch effects in single-cell RNA-sequencing data are corrected by matching mutual nearest neighbors. *Nat Biotechnol* 36:421–427.
- Korsunsky I. *et al.* (2019) Fast, sensitive and accurate integration of single-cell data with Harmony. *Nat Methods* 16:1289–1296.
- Stuart T. *et al.* (2019) Comprehensive integration of single-cell data. *Cell* 177:1888–1902.
- Hie B., Bryson B., Berger B. (2019) Efficient integration of heterogeneous single-cell transcriptomes using Scanorama. *Nat Biotechnol* 37:685–691.
- Lopez R., Regier J., Cole M. B., Jordan M. I., Yosef N. (2018) Deep generative modeling for single-cell transcriptomics. *Nat Methods* 15:1053–1058.
- Büttner M., Miao Z., Wolf F. A., Teichmann S. A., Theis F. J. (2019) A test metric for assessing single-cell RNA-seq batch correction. *Nat Methods* 16:43–49.
- Tran H. T. N. *et al.* (2020) A benchmark of batch-effect correction methods for single-cell RNA sequencing data. *Genome Biol* 21:12.
- Luecken M. D. *et al.* (2022) Benchmarking atlas-level data integration in single-cell genomics. *Nat Methods* 19:41–50.
- Lotfollahi M. *et al.* (2022) Mapping single-cell data to reference atlases by transfer learning. *Nat Biotechnol* 40:121–130.
- Xu C., Lopez R., Mehlman E., Regier J., Jordan M. I., Yosef N. (2021) Probabilistic harmonization and annotation of single-cell transcriptomics data with deep generative models. *Mol Syst Biol* 17:e9620.
- Hao Y. *et al.* (2021) Integrated analysis of multimodal single-cell data. *Cell* 184:3573–3587.
- Hubert L., Arabie P. (1985) Comparing partitions. *J Classif* 2:193–218 (adjusted Rand index).
- Kang H. M. *et al.* (2018) Multiplexed droplet single-cell RNA-sequencing using natural genetic variation. *Nat Biotechnol* 36:89–94.

## ➡️ Next

Integration aligns cells measured at one moment. Cells, however, change — they differentiate, respond and divide.
**[Module 32 · Cell dynamics: pseudotime, RNA velocity and optimal transport](32_cell_dynamics_velocity.ipynb)**
asks how to recover those dynamics from snapshots.